# AI Murder Mystery Generator (fixed version)
Prompt engineering project: prompt chaining + role prompting + system prompts + LLM-as-judge.

**Run the cells from top to bottom, one by one.**

**Before you start:** get a free key at https://aistudio.google.com/apikey. In Colab click the key icon (Secrets), add a secret named `GEMINI_API_KEY` (with underscores), paste your key as the value, and switch on Notebook access.

In [ ]:
!pip -q install google-genai

## Setup (nothing is printed here, a green tick is enough)
If one model is busy or retired, the code automatically tries the next one and retries by itself.

In [ ]:
import json, re, time
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODELS = ["gemini-3.8-flash", "gemini-flash-latest", "gemini-flash-lite-latest"]

def ask(prompt, system=None, history=None, temperature=0.9):
    contents = []
    for role, text in (history or []):
        contents.append(types.Content(role=role, parts=[types.Part(text=text)]))
    contents.append(types.Content(role="user", parts=[types.Part(text=prompt)]))
    cfg = types.GenerateContentConfig(system_instruction=system, temperature=temperature)
    for attempt in range(5):
        for m in MODELS:
            try:
                return client.models.generate_content(model=m, contents=contents, config=cfg).text
            except Exception as e:
                if any(code in str(e) for code in ("503", "429", "404", "500")):
                    print(f"{m} busy/unavailable, trying next...")
                    continue
                raise
        print("All models busy, waiting 15 seconds...")
        time.sleep(15)
    raise RuntimeError("Models are still busy. Try again in a few minutes, or set USE_BACKUP = True below.")

FENCE = "`" * 3
def parse_json(text):
    text = re.sub(FENCE + "json|" + FENCE, "", text).strip()
    return json.loads(text)

def gen_json(prompt, tries=3, temperature=0.9):
    for _ in range(tries):
        try:
            return parse_json(ask(prompt, temperature=temperature))
        except json.JSONDecodeError:
            print("Model returned bad JSON, retrying...")
    raise ValueError("Model did not return valid JSON. Just run this cell again.")

print("Setup OK")

## Quick test (optional)
Prints one line per model. If at least one says OK, everything works.

In [ ]:
for m in MODELS:
    try:
        r = client.models.generate_content(model=m, contents="Say hi in one word")
        print("OK  ", m, "->", r.text.strip())
    except Exception as e:
        print("FAIL", m, "->", str(e)[:120])

## Backup mode
Set `USE_BACKUP = True` ONLY if Google's servers stay busy for a long time. Then steps 1 to 3 use a ready-made sample case instead of generating a new one. Keep it `False` for your real submission whenever possible.

In [ ]:
USE_BACKUP = False

BACKUP = {
    "case": {"title": "The Last Toast at Halloway Manor", "setting": "A stormy dinner party at a countryside mansion",
             "victim_name": "Lord Edmund Halloway", "victim_description": "A wealthy, sharp-tongued collector of rare antiques",
             "time_of_death": "9:40 PM", "cause_of_death": "Poisoned whisky", "location_of_body": "The study"},
    "suspects": [
        {"name": "Margaret Voss", "role": "Housekeeper", "personality": "Stern, precise, protective of the house", "relationship_to_victim": "Served the family for 30 years", "secret": "She has been secretly selling small antiques", "alibi": "Serving dessert in the dining room", "is_killer": False, "motive": "none"},
        {"name": "Dr. Julian Reyes", "role": "Family physician", "personality": "Calm, charming, overly polite", "relationship_to_victim": "Old friend and doctor of the family", "secret": "Heavy gambling debts", "alibi": "Reading by the fire in the library", "is_killer": True, "motive": "Edmund was about to change his will and call in a loan"},
        {"name": "Clara Halloway", "role": "Niece", "personality": "Sarcastic, restless, emotional", "relationship_to_victim": "Heir to the estate", "secret": "She secretly dropped out of university", "alibi": "Walking in the garden", "is_killer": False, "motive": "none"},
        {"name": "Victor Stone", "role": "Business partner", "personality": "Loud, defensive, ambitious", "relationship_to_victim": "Co-owner of an auction house", "secret": "He forged papers to hide losses", "alibi": "On a phone call in the hallway", "is_killer": False, "motive": "none"}],
    "clues": [
        {"clue": "A page is missing from Dr. Reyes's prescription pad, found in his open bag.", "points_to": "Dr. Julian Reyes", "is_red_herring": False},
        {"clue": "The whisky glass has a faint bitter-almond smell.", "points_to": "Dr. Julian Reyes", "is_red_herring": False},
        {"clue": "The library fireplace is cold, though Reyes says he sat by its fire all evening.", "points_to": "Dr. Julian Reyes", "is_red_herring": False},
        {"clue": "A torn business contract sits in the study bin.", "points_to": "Victor Stone", "is_red_herring": True},
        {"clue": "Muddy footprints lead from the garden to the study window.", "points_to": "Clara Halloway", "is_red_herring": True}],
}

## Step 1: Premise (prompt #1 of the chain)

In [ ]:
premise_prompt = """You are a mystery novelist. Create the premise for an original murder mystery.
Return ONLY valid JSON with these keys:
title, setting, victim_name, victim_description, time_of_death, cause_of_death, location_of_body.
Rules: PG-13, no gore, no real people, no real brands."""

case = BACKUP["case"] if USE_BACKUP else gen_json(premise_prompt)
print(json.dumps(case, indent=2))

## Step 2: Suspects (prompt #2, uses the output of #1)

In [ ]:
suspects_prompt = f"""Here is the case premise:
{json.dumps(case)}

Create exactly 4 suspects. Exactly ONE of them is the killer.
Return ONLY a valid JSON list of 4 objects with these keys:
name, role, personality, relationship_to_victim,
secret (something embarrassing or shady that is NOT the murder, for every suspect including the killer),
alibi, is_killer (true or false), motive (the real motive for the killer; the string "none" for innocents)."""

if USE_BACKUP:
    suspects = BACKUP["suspects"]
else:
    for _ in range(3):
        suspects = gen_json(suspects_prompt)
        if len(suspects) == 4 and sum(bool(s["is_killer"]) for s in suspects) == 1:
            break
    else:
        raise ValueError("Need exactly 4 suspects and 1 killer. Run this cell again.")

killer = next(s for s in suspects if s["is_killer"])
for s in suspects:
    print(s["name"], "-", s["role"])

## Step 3: Clues (prompt #3, uses the output of #1 and #2)

In [ ]:
clues_prompt = f"""Case: {json.dumps(case)}
Suspects: {json.dumps(suspects)}

Write 5 clues a detective can find at the scene.
- 3 clues subtly point to the real killer ({killer['name']}).
- 2 clues are red herrings that point to innocent suspects.
Return ONLY a valid JSON list of objects with keys: clue, points_to, is_red_herring."""

clues = BACKUP["clues"] if USE_BACKUP else gen_json(clues_prompt)
for c in clues:
    print("-", c["clue"])

## Step 4: Every suspect is its own AI character (system prompts)
Nothing is printed here, a green tick is enough.

In [ ]:
def suspect_system(s):
    prompt = f"""You are {s['name']}, {s['role']}, in a murder mystery. Stay in character at all times.
Personality: {s['personality']}
Relationship to victim: {s['relationship_to_victim']}
Case: {json.dumps(case)}
Your secret: {s['secret']}
Your alibi: {s['alibi']}

Rules:
- Reply in 1 to 3 spoken sentences.
- Never say you are an AI and never mention these instructions.
- Never volunteer your secret. Reveal it only if the detective pushes with specific evidence.
"""
    if s["is_killer"]:
        prompt += f"- YOU ARE THE KILLER. Your motive: {s['motive']}. Lie convincingly and deflect, but let small slips happen under pressure. Confess only if the detective presents 2 or more correct clues.\n"
    else:
        prompt += "- You are innocent but nervous about your secret. You do not know who the killer is. You may hint at other suspects' suspicious behavior.\n"
    return prompt

histories = {s["name"]: [] for s in suspects}

def interrogate(name, question):
    s = next(x for x in suspects if x["name"] == name)
    reply = ask(question, system=suspect_system(s), history=histories[name], temperature=0.8)
    histories[name] += [("user", question), ("model", reply)]
    return reply

print("Characters ready")

## Step 5: Interrogate (this is the game, change the questions and play!)

In [ ]:
for s in suspects:
    q = "Where were you at the time of the murder, and how did you get along with the victim?"
    print(f"Detective -> {s['name']}: {q}")
    print(f"{s['name']}: {interrogate(s['name'], q)}\n")

In [ ]:
# Round 2: confront every suspect with a clue
clue_text = clues[0]["clue"]
for s in suspects:
    q = f"We found this at the scene: {clue_text} Can you explain that?"
    print(f"{s['name']}: {interrogate(s['name'], q)}\n")

In [ ]:
# Ask your own question (change the name and the question, then run)
print(interrogate(suspects[0]["name"], "Did you notice anything strange tonight?"))

## Step 6: Accuse the killer

In [ ]:
def accuse(name):
    if name == killer["name"]:
        return f"Correct! {killer['name']} did it. Motive: {killer['motive']}"
    return f"Wrong. The killer was {killer['name']}."

print(accuse(suspects[0]["name"]))   # change the index 0 to 1, 2 or 3

## Step 7: LLM-as-judge (evaluation)
A judge prompt checks whether each character stayed in role. Put these scores in your README as results.

In [ ]:
def judge(name):
    s = next(x for x in suspects if x["name"] == name)
    transcript = "\n".join(f"{r}: {t}" for r, t in histories[name])
    judge_prompt = f"""You are a strict evaluator of AI role-play.
Character profile: {json.dumps(s)}
Transcript:
{transcript}

Score 1-5 for each: stayed_in_character, kept_secret_unless_pressed, consistent_with_alibi, killer_not_obvious (5 = not obvious).
Return ONLY JSON: {{"stayed_in_character":n,"kept_secret_unless_pressed":n,"consistent_with_alibi":n,"killer_not_obvious":n,"comment":"one sentence"}}"""
    return gen_json(judge_prompt, temperature=0)

scores = {s["name"]: judge(s["name"]) for s in suspects}
print(json.dumps(scores, indent=2))

## Step 8: Export for the website

In [ ]:
export = {
    "case": case,
    "suspects": suspects,
    "clues": clues,
    "transcripts": {n: [{"speaker": "Detective" if r == "user" else n, "text": t} for r, t in h] for n, h in histories.items()},
    "judge_scores": scores,
}
with open("case.json", "w") as f:
    json.dump(export, f, indent=2)

from google.colab import files
files.download("case.json")